In [3]:
%pip install pandas openpyxl ipykernel

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

df = pd.read_excel("BreakThroughTech_training_data_1.xlsx")
df_clean = df.copy()
df.head()

,brand,sub_product_name,country,pillar,media_type,profession,specialty,campaign_start_date,campaign_end_date,unique_reach,...,post_avg_NRx,pre_NRx_avg,npi_specialty,region,NRx_lift,NRx_lift_pct,cpe,total_spend,actual_spend_per_engagement,spend_per_NRx_lift
0,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,8,undefined,2024-07-31 10:48:34.000000 UTC,2099-12-31 23:59:59.999999 UTC,9270,...,72.8343,47.4714,Neurology,Southeast,146.77,15.5000,39.0,134355308.0,30198.99,915440.73
1,ABBVIE,BRAND ALERT,usa,search,EDITORIAL_VIDEOS,8,undefined,2024-08-19 17:16:44.000000 UTC,2099-12-31 23:59:59.999999 UTC,1959,...,54.1279,62.6833,Cardiology,Southwest,112.05,12.2398,39.0,10667850.0,9055.90,95202.91
2,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,psychologist,urology,2023-11-16 15:54:10.000000 UTC,2099-12-31 23:59:59.999999 UTC,2860,...,42.1019,66.2213,Cardiology,Mid-Atlantic,78.27,10.8516,39.0,59103000.0,34705.23,755161.42
3,ABBVIE,BRAND ALERT,zaf,search,SPONSORED_VIDEOS,8,undefined,2024-11-22 10:27:31.000000 UTC,2099-12-31 23:59:59.999999 UTC,7390,...,49.5424,55.8293,Neurology,Northeast,121.87,11.9201,39.0,124487739.0,112455.05,1021479.17
4,ABBVIE,EMAIL ALERT,NaN,NaN,NaN,NaN,NaN,2023-11-16 17:38:18.000000 UTC,2099-12-31 23:59:59.999999 UTC,77570,...,46.8907,39.3346,Endocrinology,Southwest,221.51,35.1116,3.0,892409925.0,7691.40,4028833.69


Each problem is fixed in a separate code cell below. Comments explain why each change is made. The fixes are saved in df_clean so the original df stays unchanged.

In [5]:
# Issue: post_avg_NRx has 20 unusually high values
# Fix post_avg_NRx outliers by capping at  IQR upper limit -> keeps rows but prevents unusually high val. from dominating analysis

q1 = df_clean["post_avg_NRx"].quantile(0.25)
q3 = df_clean["post_avg_NRx"].quantile(0.75)
upper_limit = q3 + 1.5 * (q3 - q1)
post_outliers = df_clean["post_avg_NRx"] > upper_limit
print("Rows fixed:", post_outliers.sum())
df_clean.loc[post_outliers, "post_avg_NRx"] = upper_limit
df_clean.loc[post_outliers, ["post_avg_NRx"]].head()

Rows fixed: 20


,post_avg_NRx
71,319.3054
170,319.3054
174,319.3054
189,319.3054
192,319.3054


In [6]:
# Issue:pre_NRx_avg has 14 unusually high values
# Fix pre_NRx_avg outliers by capping at IQR upper -> same idea as post_avg_NRx
q1 = df_clean["pre_NRx_avg"].quantile(0.25)
q3 = df_clean["pre_NRx_avg"].quantile(0.75)
upper_limit = q3 + 1.5 * (q3 - q1)
pre_outliers = df_clean["pre_NRx_avg"] > upper_limit
print("Rows fixed:", pre_outliers.sum())
df_clean.loc[pre_outliers, "pre_NRx_avg"] = upper_limit
df_clean.loc[pre_outliers, ["pre_NRx_avg"]].head()

Rows fixed: 14


,pre_NRx_avg
173,232.6029
182,232.6029
189,232.6029
193,232.6029
198,232.6029


In [7]:
# Issue: npi_specialty has 1 coded value instead of a plainspecialty name
# Fix the one coded npi_specialty value -> format more consistent
npi_problem = df_clean["npi_specialty"] == "3320 - Anesthesiologists"
print("Rows fixed:", npi_problem.sum())
df_clean.loc[npi_problem, "npi_specialty"] = "Anesthesiology"
df_clean.loc[npi_problem, ["npi_specialty"]]

Rows fixed: 1


,npi_specialty
136,Anesthesiology


In [8]:
# Issue: region has 1 inconsistent label that appears only once
# Fix the one region label that appears only once
# West South Central is grouped into the existing Southwest category 
region_problem = df_clean["region"] == "West South Central"
print("Rows fixed:", region_problem.sum())
df_clean.loc[region_problem, "region"] = "Southwest"
df_clean.loc[region_problem, ["region"]]

Rows fixed: 1


,region
136,Southwest


In [9]:
# Issue: NRx_lift has many one-cent rounding differences
# Recalc. NRx_lift so it always equals post_NRx minus pre_NRx
# Rounding to two decimals removes one-cent inconsistencies
expected_lift = (df_clean["post_NRx"] - df_clean["pre_NRx"]).round(2)
lift_problem = df_clean["NRx_lift"] != expected_lift
print("Rows fixed:", lift_problem.sum())
df_clean.loc[lift_problem, "NRx_lift"] = expected_lift[lift_problem]
df_clean.loc[lift_problem, ["pre_NRx", "post_NRx", "NRx_lift"]].head()

Rows fixed: 106


,pre_NRx,post_NRx,NRx_lift
0,946.88,1093.64,146.76
5,1096.71,1279.66,182.95
10,75.20,85.11,9.91
13,66.57,85.62,19.05
15,723.08,1047.03,323.95


In [10]:
# Show cleaned column (from my data)
columns = ["post_avg_NRx", "pre_NRx_avg", "npi_specialty", "region", "NRx_lift"]
df_clean[columns].head()

,post_avg_NRx,pre_NRx_avg,npi_specialty,region,NRx_lift
0,72.8343,47.4714,Neurology,Southeast,146.76
1,54.1279,62.6833,Cardiology,Southwest,112.05
2,42.1019,66.2213,Cardiology,Mid-Atlantic,78.27
3,49.5424,55.8293,Neurology,Northeast,121.87
4,46.8907,39.3346,Endocrinology,Southwest,221.51
